# Sensitivity of filter feeders to TEP excretion (`p_pueR2`)

Suspension feeders (`Y3`, and juveniles `Yy3`) filter TEP (`R2`) from the water.
They excrete a fraction `p_pueR2` of the TEP they take up (as faeces, to `R6`)
and assimilate the rest. The OysterGrounds case uses `p_pueR2 = 0.90` (90 %).
This notebook compares that with 40 %, 20 % and 10 %.

`p_pueR2` is a namelist parameter in `FilterFeeder.nml` (`/FilterFeeder_parameters/`),
read by `src/BFM/Ben/FilterFeeder.F90` (`se_uR2 = suf*p_pueR2(y)`), so **no recompilation
is needed**. The parameter has one value per filter-feeder group (`Y3`, `Yy3`); both are
set to the same value. Do not use exactly 0: the model takes that as "no TEP uptake".

`p_pueR3` (TEP inside *Phaeocystis* colonies, `R3`) is kept at the case value 0.90 by
default. Each scenario in `param_scenarios` is a `pueR2/pueR3` pair, so it can be changed
too, e.g. `"0.90/0.90, 0.40/0.40"`.

Plots of the mean suspension-feeder biomass (`Y3c`) over one year:

* **(a)** annual mean `Y3c` per scenario, labelled with the % change from the first scenario
* **(b)** daily `Y3c` through the year, one line per scenario

One year takes about 6 minutes per run; the runs are started in parallel.

## NaaVRE workflow

Two workflow components:

```
[run GOTM TEP] --> [plot TEP]
   run_dirs: List     results_folder: String
```

1. **run GOTM TEP**: turns `param_scenarios` into one model run per scenario, runs them in
   parallel and outputs `run_dirs`, the list of run folders under `/tmp/data/` (the only
   folder shared between components).
2. **plot TEP**: reads `OG.nc` from each folder, makes the table and figure and saves them,
   with each run's `log.out` and `run_info.json`, to Cloud Storage, because NaaVRE clears
   `/tmp/data/` when the workflow finishes. It outputs that folder as `results_folder`.

In [ ]:
# run GOTM TEP
# ---
# NaaVRE:
#  cell:
#   inputs: []
#   outputs:
#    - run_dirs: List
#   params:
#    - param_year:
#       type: Integer
#       default_value: 1972
#    - param_var:
#       type: String
#       default_value: "Y3c"
#    - param_start:
#       type: String
#       default_value: "1972-01-01 00:00:00"
#    - param_stop:
#       type: String
#       default_value: "1973-01-01 00:00:00"
#    - param_scenarios:
#       type: String
#       default_value: "0.90/0.90, 0.40/0.90, 0.20/0.90, 0.10/0.90"
# ...


# Settings (do not containerize this cell)
# param_ values are the defaults of the workflow components; they can be changed when
# the workflow is submitted.
param_scenarios = "0.90/0.90, 0.40/0.90, 0.20/0.90, 0.10/0.90"   # pueR2/pueR3 per run; the first is the reference
param_start = "1972-01-01 00:00:00"
param_stop = "1973-01-01 00:00:00"
param_results_root = "/tmp/data/TEPexcretion"   # shared between workflow components
param_year = 1972            # year to average over
param_var = "Y3c"            # BFM variable in OG.nc, e.g. "jnetY3c" for net production
param_cloud_folder = "/home/jovyan/Cloud Storage/naa-vre-user-data/TEPexcretion"   # kept after the workflow

# TEP scenarios: labels "pueR2_<x>_pueR3_<y>", one per model run
scenarios = []
for pair in str(param_scenarios).split(","):
    if not pair.strip():
        continue
    pueR2, pueR3 = [float(v) for v in pair.split("/")]
    scenarios.append("pueR2_{:.2f}_pueR3_{:.2f}".format(pueR2, pueR3))
print("scenarios:", scenarios)

# run GOTM TEP
# Output run_dirs (List of str): run folders under param_results_root.
import os
import re
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor


def run_one(label):
    """One GOTM-BFM run for a label "pueR2_<x>_pueR3_<y>"; returns the run folder."""
    parts = label.split("_")
    pueR2, pueR3 = float(parts[1]), float(parts[3])
    run_dir = os.path.join(param_results_root, label)
    if os.path.exists(run_dir):          # gotm-run needs a new or empty directory
        shutil.rmtree(run_dir)
    os.makedirs(param_results_root, exist_ok=True)
    command = [
        "gotm-run",
        "--out", run_dir,
        "--start", param_start,
        "--stop", param_stop,
        # one value per filter-feeder group (Y3, Yy3)
        "--set", "FilterFeeder.nml:p_pueR2={0:g},{0:g}".format(pueR2),
        "--set", "FilterFeeder.nml:p_pueR3={0:g},{0:g}".format(pueR3),
    ]
    proc = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    status = "finished" if proc.returncode == 0 else "FAILED (see {}/log.out)".format(run_dir)
    # namelist echo in log.out: the values the model actually used
    used = ""
    log_file = os.path.join(run_dir, "log.out")
    if os.path.exists(log_file):
        with open(log_file, errors="replace") as f:
            used = " | ".join(" ".join(u.split()) for u in
                              re.findall(r"P_PUER[23]\s*=[^A-Z]*", f.read(), flags=re.IGNORECASE))
    print("{:<24} {}   {}".format(label, status, used or "(values not found in log.out)"), flush=True)
    if proc.returncode != 0:
        print(proc.stdout.decode(errors="replace")[-2000:])
    return run_dir


print("Starting {} run(s) from {} to {} ...".format(len(scenarios), param_start, param_stop), flush=True)
with ThreadPoolExecutor(max_workers=max(1, len(scenarios))) as pool:
    run_dirs = list(pool.map(run_one, scenarios))
print("run_dirs:", run_dirs)

In [ ]:
# plot TEP
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - run_dirs: List
#   outputs:
#    - results_folder: String
# ...

# plot TEP
# Input run_dirs (List of str): run folders from the run cell.
# Output results_folder (str): Cloud Storage folder with the figure and the table.
import datetime
import json
import os
import shutil

import matplotlib.pyplot as plt
import netCDF4
import numpy as np

# Flatten, in case run_dirs arrives as a list of lists
folders = []
for item in (run_dirs if isinstance(run_dirs, (list, tuple)) else [run_dirs]):
    folders.extend(item if isinstance(item, (list, tuple)) else [item])

year = int(param_year)
var = str(param_var)

results = []    # (pueR2, pueR3, dates, values)
units, long_name = "", var
for run_dir in folders:
    label = os.path.basename(os.path.normpath(run_dir))
    info_file = os.path.join(run_dir, "run_info.json")
    nc_file = os.path.join(run_dir, "OG.nc")
    if os.path.exists(info_file):
        with open(info_file) as f:
            if json.load(f).get("status") != "finished":
                print("skipped (run failed):", run_dir)
                continue
    if not os.path.exists(nc_file):
        print("skipped (no OG.nc):", run_dir)
        continue
    parts = label.split("_")              # pueR2_<x>_pueR3_<y>
    pueR2, pueR3 = float(parts[1]), float(parts[3])
    with netCDF4.Dataset(nc_file) as nc:
        t = nc.variables["time"]
        # num2date may return cftime dates, which matplotlib cannot plot:
        # convert them to standard Python datetimes
        dates = [datetime.datetime(d.year, d.month, d.day, d.hour, d.minute, d.second)
                 for d in netCDF4.num2date(t[:], t.units)]
        values = np.ma.filled(nc.variables[var][:].reshape(len(dates), -1)[:, 0], np.nan)
        units = getattr(nc.variables[var], "units", "")
        long_name = getattr(nc.variables[var], "long_name", var)
    keep = np.array([d.year == year for d in dates])
    results.append((pueR2, pueR3, [d for d, k in zip(dates, keep) if k], values[keep]))
if not results:
    raise RuntimeError("no finished runs in {}".format(folders))
results.sort(key=lambda r: (-r[0], -r[1]))     # from high to low excretion

# Annual mean and % change from the case default (pueR2 = pueR3 = 0.90, or the first run)
labels = ["R2 {:.2f}, R3 {:.2f}".format(r[0], r[1]) for r in results]
means = np.array([np.nanmean(r[3]) for r in results])
i_ref = next((i for i, r in enumerate(results)
              if abs(r[0] - 0.9) < 1e-6 and abs(r[1] - 0.9) < 1e-6), 0)
ref = means[i_ref]
change = 100.0 * (means - ref) / ref

table = ["p_pueR2,p_pueR3,mean_{},change_percent".format(var)]
print("reference:", labels[i_ref])
print("{:>7}  {:>7}  {:>12}  {:>8}".format("p_pueR2", "p_pueR3", "mean " + var, "change"))
for r, m, c in zip(results, means, change):
    print("{:>7.2f}  {:>7.2f}  {:>12.2f}  {:>+7.2f}%".format(r[0], r[1], m, c))
    table.append("{:g},{:g},{:.6g},{:.4f}".format(r[0], r[1], m, c))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
colors = plt.cm.viridis(np.linspace(0, 0.9, len(results)))
x = np.arange(len(results))

ax1.bar(x, means, color=colors, width=0.6)
for xi, m, c in zip(x, means, change):
    ax1.annotate("{:+.1f}%".format(c), (xi, m), textcoords="offset points",
                 xytext=(0, 4), ha="center", fontsize=9)
ax1.set_xticks(x)
ax1.set_xticklabels([l.replace(", ", "\n") for l in labels], fontsize=9)
ax1.set_xlabel("excreted fraction of TEP uptake (p_pueR2, p_pueR3)")
ax1.set_ylabel("annual mean {} [{}]".format(var, units))
ax1.set_title("(a) {}, mean over {}".format(long_name, year), fontsize=10)
ax1.margins(y=0.12)

for (pueR2, pueR3, dates, values), col, l in zip(results, colors, labels):
    ax2.plot(dates, values, color=col, label=l)
ax2.set_ylabel("{} [{}]".format(var, units))
ax2.set_title("(b) daily values in {}".format(year), fontsize=10)
ax2.legend(title="scenario", fontsize=8, frameon=False)
fig.autofmt_xdate()
plt.setp(ax1.get_xticklabels(), rotation=0, ha="center")   # keep scenario labels upright
fig.tight_layout()

# Save to Cloud Storage: /tmp/data is cleared when the workflow finishes
results_folder = str(param_cloud_folder)
os.makedirs(results_folder, exist_ok=True)
figure_file = os.path.join(results_folder, "TEPexcretion_{}_{}.png".format(var, year))
fig.savefig(figure_file, dpi=150)
with open(os.path.join(results_folder, "TEPexcretion_{}_{}.csv".format(var, year)), "w") as f:
    f.write("\n".join(table) + "\n")
for run_dir in folders:                  # keep the model logs and run info as well
    name = os.path.basename(os.path.normpath(run_dir))
    for fname in ("log.out", "run_info.json"):
        if os.path.exists(os.path.join(run_dir, fname)):
            os.makedirs(os.path.join(results_folder, name), exist_ok=True)
            shutil.copy(os.path.join(run_dir, fname), os.path.join(results_folder, name, fname))
print("results_folder:", results_folder)
print("figure:", figure_file)
plt.show()